# Install and import librairies

In [3]:
import sys
import os

def install_openpyxl():
    try:
        import openpyxl
        print("openpyxl est déjà installé.")
    except ImportError:
        print("Installation de openpyxl en cours...")
        os.system(f"{sys.executable} -m pip install openpyxl")
        print("Installation terminée.")

if __name__ == "__main__":
    install_openpyxl()


openpyxl est déjà installé.


In [4]:
import os
import re
from openpyxl import Workbook


# Extract the Informations (filename and company name) to put it in the excel later

In [5]:
import os
import re

def extract_info_from_file(file_path):
    """
    Extrait les informations <FileName>...<FileName> et 'COMPANY CONFORMED NAME'
    d'un fichier texte.

    Retourne un tuple (filename_tag, company_name).
    Si une info n'est pas trouvée ou qu'une erreur survient, retourne des chaînes vides.
    """
    filename_pattern = re.compile(r"<FileName>(.*?)</FileName>")
    filename_tag = ""
    company_name = ""

    if not os.path.isfile(file_path):
        return "", ""

    try:
        with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
            for line in f:
                match_filename = filename_pattern.search(line)
                if match_filename and not filename_tag:
                    filename_tag = match_filename.group(1).strip()
                if "COMPANY CONFORMED NAME:" in line and not company_name:
                    parts = line.split("COMPANY CONFORMED NAME:")
                    if len(parts) > 1:
                        company_name = parts[1].strip()
    except Exception as e:
        filename_tag = ""
        company_name = ""
    return filename_tag, company_name


In [9]:
data_folder = "data"
results = []
for root, dirs, files in os.walk(data_folder):
    for file_name in files:
        if file_name.lower().endswith(".txt"):
            file_path = os.path.join(root, file_name)
            filename_tag, company_name = extract_info_from_file(file_path)
            print(f"Filename: {filename_tag}")
            if filename_tag or company_name:
                results.append({
                    "filename_tag": filename_tag,
                    "company_name": company_name
                })


Filename: 20230103_10-K-A_edgar_data_1880151_0001104659-22-131423.txt
Filename: 20230103_10-K_edgar_data_1487931_0001477932-23-000012.txt
Filename: 202301z03_10-K_edgar_data_1828739_0001477932-23-000002.txt
Filename: 20230104_10-K-A_edgar_data_1889450_0001493152-23-000281.txt
Filename: 20230104_10-K_edgar_data_715446_0001493152-23-000346.txt
Filename: 20230105_10-K_edgar_data_1673431_0001096906-23-000016.txt
Filename: 20230106_10-K-A_edgar_data_1726126_0001558370-23-000085.txt
Filename: 20230106_10-K_edgar_data_1619096_0001091818-23-000002.txt
Filename: 20230106_10-K_edgar_data_1620749_0001640334-23-000027.txt
Filename: 20230106_10-K_edgar_data_315374_0001558370-23-000097.txt


# Processing the data to remove useless data

In [ ]:
import os
import re
import shutil

# Remove all 'processed_*' folders before starting
base_dir = "./data"
for root, dirs, files in os.walk(base_dir):
    for dir_name in dirs:
        if dir_name.startswith("processed_"):
            shutil.rmtree(os.path.join(root, dir_name))
            print(f"Deleted folder: {os.path.join(root, dir_name)}")

# Traverse files and process each text file
for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.endswith(".txt"):
            input_path = os.path.join(root, file)

            # Create the output directory
            relative_dir = os.path.relpath(root, base_dir)
            processed_dir = os.path.join(base_dir, f"processed_{relative_dir}")
            os.makedirs(processed_dir, exist_ok=True)
            output_path = os.path.join(processed_dir, file)

            try:
                with open(input_path, "r", encoding="utf-8") as f:
                    content = f.read()
            except FileNotFoundError:
                print(f"File not found: {input_path}. Skipped.")
                continue

            lower_content = content.lower()

            # First extraction: ITEM 1A RISK FACTORS -> ITEM 1B UNRESOLVED STAFF
            start_pattern_1 = r"item\s*1\s*a\.?\s*risk factors"
            end_pattern_1 = r"item\s*1\s*b\.?\s*unresolved staff"

            start_match_1 = list(re.finditer(start_pattern_1, lower_content))
            end_match_1 = list(re.finditer(end_pattern_1, lower_content))

            if not start_match_1 or not end_match_1:
                print(f"No section found between 'ITEM 1A' and 'ITEM 1B' in {input_path}. Skipped.")
                continue

            start_idx_1 = start_match_1[-1].end()
            end_idx_1 = end_match_1[-1].start()

            if start_idx_1 > end_idx_1:
                print(f"The section 'ITEM 1A RISK FACTORS' appears after 'ITEM 1B UNRESOLVED STAFF' in {input_path}. Skipped.")
                continue

            extracted_text_1 = content[start_idx_1:end_idx_1].strip()

            # Second extraction: ITEM 7 -> ITEM 7A or ITEM 8
            start_pattern_2 = r"item\s*7\.?\s*management\s*s\s*discussion\s*and\s*analysis\s*of\s*financial\s*condition\s*and\s*results\s*of\s*operations"
            end_pattern_2_primary = r"item\s*7a\.?\s*quantitative\s*and\s*qualitative"
            end_pattern_2_secondary = r"item\s*8\.?\s*financial\s*statements\s*and\s*supplementary\s*data"

            start_match_2 = list(re.finditer(start_pattern_2, lower_content))
            end_match_2_primary = list(re.finditer(end_pattern_2_primary, lower_content))
            end_match_2_secondary = list(re.finditer(end_pattern_2_secondary, lower_content))

            if not start_match_2:
                print(f"No section found for 'ITEM 7 MANAGEMENT...' in {input_path}. Skipped.")
                continue

            start_idx_2 = start_match_2[-1].end()
            if end_match_2_primary:
                end_idx_2 = end_match_2_primary[-1].start()
            elif end_match_2_secondary:
                end_idx_2 = end_match_2_secondary[-1].start()
            else:
                print(f"No ending found for 'ITEM 7 MANAGEMENT...' in {input_path}. Skipped.")
                continue

            if start_idx_2 > end_idx_2:
                print(f"The section 'ITEM 7 MANAGEMENT...' appears after 'ITEM 7A...' or 'ITEM 8...' in {input_path}. Skipped.")
                continue

            extracted_text_2 = content[start_idx_2:end_idx_2].strip()

            # Merge results
            final_extracted_text = extracted_text_1 + "\n\n" + extracted_text_2

            # Remove lines containing only numbers and spaces
            final_extracted_text = "\n".join(
                line for line in final_extracted_text.splitlines()
                if not re.match(r"^\s*\d+\s*$", line)
            )

            # Save the result
            with open(output_path, "w", encoding="utf-8") as out:
                out.write(final_extracted_text)

            print(f"Extraction completed for {input_path}. Result saved in {output_path}.")

Deleted folder: ./data\processed_QTR1,
Deleted folder: ./data\processed_QTR2,
Deleted folder: ./data\processed_QTR3,
Deleted folder: ./data\processed_QTR4,
No section found for 'ITEM 7 MANAGEMENT...' in ./data\QTR1,\20230103_10-K-A_edgar_data_1880151_0001104659-22-131423.txt. Skipped.
Extraction completed for ./data\QTR1,\20230103_10-K_edgar_data_1487931_0001477932-23-000012.txt. Result saved in ./data\processed_QTR1,\20230103_10-K_edgar_data_1487931_0001477932-23-000012.txt.
No section found for 'ITEM 7 MANAGEMENT...' in ./data\QTR1,\20230103_10-K_edgar_data_1828739_0001477932-23-000002.txt. Skipped.
No section found for 'ITEM 7 MANAGEMENT...' in ./data\QTR1,\20230104_10-K-A_edgar_data_1889450_0001493152-23-000281.txt. Skipped.
Extraction completed for ./data\QTR1,\20230104_10-K_edgar_data_715446_0001493152-23-000346.txt. Result saved in ./data\processed_QTR1,\20230104_10-K_edgar_data_715446_0001493152-23-000346.txt.
No section found between 'ITEM 1A' and 'ITEM 1B' in ./data\QTR1,\202

# Create the excel

In [7]:
def create_excel(data_list, output_filename="output.xlsx"):
    """
    Crée un fichier Excel avec trois colonnes :
    Filename, Company name, AI Probability
    """
    wb = Workbook()
    ws = wb.active
    ws.title = "Data"
    ws.append(["Filename", "Company Name", "AI Probability"])

    for item in data_list:
        filename_tag = item["filename_tag"]
        company_name = item["company_name"]
        ws.append([filename_tag, company_name, ""])

    wb.save(output_filename)
    print(f"Fichier Excel généré : {output_filename}")


In [8]:
create_excel(results, output_filename="output.xlsx")

Fichier Excel généré : output.xlsx
